# EmbedTrack: pretrained-model SEG / TRA on Fluo-N2DH-GOWT1 and Fluo-N2DL-HeLa

Runs the **pretrained KIT-Loe-GE models** (the ones submitted to the Cell Tracking Challenge) on the
CTC **training sequences 01 and 02** of two 2D datasets, scores the results with the official CTC
measures (SEG, TRA, DET), and compares them with the numbers published in the EmbedTrack paper.

**Read before quoting numbers.** The paper's scores come from the hidden *challenge* sequences, which
nobody can score locally. The scores below are on the *training* sequences, and the pretrained models were
trained on 90% of those frames, so they are expected to be optimistic. Report them as
"published value vs. reproduced value on training sequences", not as an exact replication.

Run this inside a Slurm GPU job (see the run steps), not on the login node.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
from time import time

PROJECT = Path.home() / "clinic-2026-cell-tracking" / "external" / "EmbedTrack"
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU found - run this inside a Slurm GPU job"

from embedtrack.ctc_metrics.eval_ctc import calc_ctc_scores
from embedtrack.infer.infer_ctc_data import inference

In [ ]:
DATA_SETS = ["Fluo-N2DH-GOWT1"]
SEQUENCES = ["01", "02"]
BATCH_SIZE = 32
RUN_TAG = "KIT-Loe-GE"

RAW = PROJECT / "ctc_raw_data" / "train"              # symlink to the shared CTC folder
MODEL_ROOT = PROJECT / "KIT-Loe-GE" / "models"       # from KIT-Loe-GE.zip
RES = PROJECT / "results"

# Table 1 of Loeffler & Mikut (challenge sequences). OP_CTB = (SEG + TRA) / 2.
PAPER = {
    "Fluo-N2DH-GOWT1": {"SEG": 0.929, "TRA": 0.951, "OP_CTB": 0.940},
    "Fluo-N2DL-HeLa":  {"SEG": 0.906, "TRA": 0.992, "OP_CTB": 0.949},
}

In [ ]:
problems = []
if not (PROJECT / "embedtrack" / "ctc_metrics" / "CTC_eval").exists():
    problems.append("missing embedtrack/ctc_metrics/CTC_eval (unzip EvaluationSoftware.zip there)")
for ds in DATA_SETS:
    for seq in SEQUENCES:
        for p in (RAW / ds / seq, Path(str(RAW / ds / seq) + "_GT")):
            if not p.exists():
                problems.append(f"missing {p}")
    for f in ("best_iou_model.pth", "config.json"):
        if not (MODEL_ROOT / ds / f).exists():
            problems.append(f"missing {MODEL_ROOT / ds / f}")
print("\n".join(problems) if problems else "All inputs found.")
assert not problems, "fix the items above, then re-run"

## Can the official scoring tools run here?
The CTC `Linux/*Measure` programs crashed (segmentation fault) on the login node in earlier tests. This cell
runs one of them on the sample data that ships with the tools. If it fails, the notebook still runs inference
and keeps the results, but skips scoring, so nothing is left half-moved by a crashing scorer.

In [ ]:
import subprocess

def ctc_tools_work():
    exe = PROJECT / "embedtrack" / "ctc_metrics" / "CTC_eval" / "Linux" / "SEGMeasure"
    sample = PROJECT / "embedtrack" / "ctc_metrics" / "CTC_eval" / "testing_dataset"
    r = subprocess.run([str(exe), str(sample), "01", "3"], capture_output=True, text=True)
    print("return code:", r.returncode)
    print("stdout:", r.stdout.strip()[:300])
    print("stderr:", r.stderr.strip()[:300])
    return r.returncode == 0

SCORE = ctc_tools_work()
print("Scoring enabled:", SCORE)
if not SCORE:
    print("WARNING: the CTC scoring tools do not run on this node. Inference will still run; "
          "score the saved results later (e.g. on another machine).")

## Inference and scoring
`inference` writes `<sequence>_RES` next to the raw data (inside the shared dataset folder), and the notebook
moves it into `results/`. If a results folder already exists it is reused, so the notebook can be re-run safely.

In [ ]:
scores = {}
for ds in DATA_SETS:
    model_path = MODEL_ROOT / ds / "best_iou_model.pth"
    config_file = MODEL_ROOT / ds / "config.json"
    for seq in SEQUENCES:
        img_path = RAW / ds / seq
        res_path = RES / ds / RUN_TAG / "train" / f"{seq}_RES"
        if res_path.exists():
            print("reusing existing results:", res_path)
        else:
            t0 = time()
            inference(str(img_path), str(model_path), str(config_file), batch_size=BATCH_SIZE)
            print(f"{ds} {seq}: inference took {time() - t0:.0f} s")
            res_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(img_path) + "_RES", str(res_path))
        if SCORE:
            metrics = calc_ctc_scores(res_path, Path(str(img_path) + "_GT"))
            scores[f"{ds}/{seq}"] = metrics
            print(ds, seq, metrics)
        else:
            print(f"{ds} {seq}: scoring skipped; results kept at {res_path}")

In [ ]:
out = RES / f"ctc_scores_{RUN_TAG}_pretrained.json"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w") as fh:
    json.dump(scores, fh, indent=2, default=str)
print("saved", out)

if not scores:
    print()
    print("No scores were computed (scoring tools unavailable). Inference results are saved here:")
    for ds in DATA_SETS:
        for seq in SEQUENCES:
            print("  results:", RES / ds / RUN_TAG / "train" / f"{seq}_RES")
            print("  ground truth:", Path(str(RAW / ds / seq) + "_GT"))

print()
print(f"{'dataset':18s}{'seq':>5s}{'SEG':>8s}{'TRA':>8s}{'DET':>8s}")
for key, m in scores.items():
    ds, seq = key.split("/")
    print(f"{ds:18s}{seq:>5s}{m.get('SEG', float('nan')):8.3f}{m.get('TRA', float('nan')):8.3f}{m.get('DET', float('nan')):8.3f}")

print()
print("Mean over sequences vs. paper (paper = challenge sequences, ours = training sequences)")
print(f"{'dataset':18s}{'metric':>8s}{'paper':>9s}{'ours':>9s}{'diff':>9s}")
for ds in DATA_SETS:
    if not all(f"{ds}/{s}" in scores for s in SEQUENCES):
        print(f"{ds}: no scores to compare")
        continue
    seg = sum(scores[f"{ds}/{s}"]["SEG"] for s in SEQUENCES) / len(SEQUENCES)
    tra = sum(scores[f"{ds}/{s}"]["TRA"] for s in SEQUENCES) / len(SEQUENCES)
    ours = {"SEG": seg, "TRA": tra, "OP_CTB": (seg + tra) / 2}
    for k in ("SEG", "TRA", "OP_CTB"):
        print(f"{ds:18s}{k:>8s}{PAPER[ds][k]:9.3f}{ours[k]:9.3f}{ours[k] - PAPER[ds][k]:+9.3f}")

## Quick visual check (raw frame vs. predicted labels)
Saves PNGs next to the results so they can be copied to a laptop.

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt

for ds in DATA_SETS:
    seq = "01"
    res_dir = RES / ds / RUN_TAG / "train" / f"{seq}_RES"
    masks = sorted(res_dir.glob("mask*.tif"))
    if not masks:
        print("no masks found in", res_dir)
        continue
    for idx in (0, len(masks) // 2, len(masks) - 1):
        n = int(masks[idx].stem.replace("mask", ""))
        raw_file = RAW / ds / seq / f"t{n:03d}.tif"
        img = tifffile.imread(raw_file)
        lab = tifffile.imread(masks[idx])
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))
        ax[0].imshow(img, cmap="gray")
        ax[0].set_title(f"{ds} seq {seq} raw t{n:03d}")
        shown = np.ma.masked_where(lab == 0, lab % 20)
        ax[1].imshow(img, cmap="gray")
        ax[1].imshow(shown, cmap="tab20", alpha=0.6, interpolation="nearest")
        ax[1].set_title(f"predicted labels ({len(np.unique(lab)) - 1} cells)")
        for a in ax:
            a.axis("off")
        png = res_dir.parent / f"preview_{seq}_t{n:03d}.png"
        fig.savefig(png, dpi=120, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print("saved", png)